In [1]:
import hdbscan
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import warnings
from umap import UMAP

warnings.filterwarnings("ignore", category=FutureWarning)

c:\Users\Usuario\anaconda3\envs\reto10_pyg\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('datos/df_final.csv', header=1)
df_covid = df[df['DIAG ING/INPAT'].str.contains('COVID19', na=False, case=False)]
df_covid_dia = df_covid.filter(regex='^DIA_')
df_covid_dia.loc[:, 'COMORBILIDADES/COMORBILITIES'] = df_covid_dia.apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)
# unir la columna al dataframe original df_covid
df_covid = df_covid.join(df_covid_dia['COMORBILIDADES/COMORBILITIES'])
df_covid['COMORBILIDADES/COMORBILITIES'].value_counts().head(10)
columnas = ['PATIENT ID', 'EDAD/AGE', 'SEXO/SEX', 'DIAG ING/INPAT',
                'FARMACO/DRUG_NOMBRE_COMERCIAL/COMERCIAL_NAME', 'UCI_DIAS/ICU_DAYS',
                'TEMP_ING/INPAT', 'SAT_02_ING/INPAT', 'RESULTADO/VAL_RESULT',
                'MOTIVO_ALTA/DESTINY_DISCHARGE_ING']
    
df_covid = df_covid[columnas]
# Rellenar los valores nulos con 0
df_covid.fillna(0, inplace=True)
# Convertir las columnas numéricas a tipo float
# Convertir las columnas a numéricas, forzando errores a NaN y luego rellenando con 0 antes de convertir a int/float
df_covid['EDAD/AGE'] = pd.to_numeric(df_covid['EDAD/AGE'], errors='coerce').fillna(0).astype(int)
df_covid['PATIENT ID'] = pd.to_numeric(df_covid['PATIENT ID'], errors='coerce').fillna(0).astype(int)
df_covid['UCI_DIAS/ICU_DAYS'] = pd.to_numeric(df_covid['UCI_DIAS/ICU_DAYS'], errors='coerce').fillna(0).astype(int)
df_covid['TEMP_ING/INPAT'] = pd.to_numeric(df_covid['TEMP_ING/INPAT'], errors='coerce').fillna(0).astype(float)
df_covid['SAT_02_ING/INPAT'] = pd.to_numeric(df_covid['SAT_02_ING/INPAT'], errors='coerce').fillna(0).astype(int)
# imputar los valores 0 por la media de la columna
df_covid['UCI_DIAS/ICU_DAYS'] = df_covid['UCI_DIAS/ICU_DAYS'].replace(0, round(df_covid['UCI_DIAS/ICU_DAYS'].mean()))
df_covid['EDAD/AGE'] = df_covid['EDAD/AGE'].replace(0, round((df_covid['EDAD/AGE'].mean())))
df_covid['TEMP_ING/INPAT'] = df_covid['TEMP_ING/INPAT'].replace(0, round(df_covid['TEMP_ING/INPAT'].mean()))
df_covid['PATIENT ID'] = df_covid['PATIENT ID'].replace(0, round(df_covid['PATIENT ID'].mean()))
df_covid['SAT_02_ING/INPAT'] = df_covid['SAT_02_ING/INPAT'].replace(0, round(df_covid['SAT_02_ING/INPAT'].mean()))
# en MOTIVO_ALTA/DESTINY_DISCHARGE_ING sustituir los valores nulos por 'Domicilio'
df_covid['MOTIVO_ALTA/DESTINY_DISCHARGE_ING'] = df_covid['MOTIVO_ALTA/DESTINY_DISCHARGE_ING'].fillna('Domicilio')

C:\Users\Usuario\AppData\Local\Temp\ipykernel_6768\1844915549.py:1: DtypeWarning: Columns (5,6,7,9,10,11,12,15,20,23,26,28,32,37,39,41,42,43,44,126) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('datos/df_final.csv', header=1)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_6768\1844915549.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_covid_dia.loc[:, 'COMORBILIDADES/COMORBILITIES'] = df_covid_dia.apply(lambda x: ', '.join(x.dropna().astype(str)), axis=1)


In [3]:
df_synt = pd.read_csv('datos/datos_sinteticos_tvae.csv', index_col=0)

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier

In [5]:
numeric_cols = ['EDAD/AGE', 'UCI_DIAS/ICU_DAYS', 'TEMP_ING/INPAT', 'SAT_02_ING/INPAT']
desc_real = df_covid[numeric_cols].describe().T[['mean', 'std']]
desc_synth = df_synt[numeric_cols].describe().T[['mean', 'std']]
comparison_stats = desc_real.join(desc_synth, lsuffix='_real', rsuffix='_synth')
print("Descriptive Statistics Comparison:\n", comparison_stats)

Descriptive Statistics Comparison:
                    mean_real   std_real  mean_synth  std_synth
EDAD/AGE           67.840036  16.048206   67.652000  17.404937
UCI_DIAS/ICU_DAYS   1.478105   2.789159    6.512000   2.284530
TEMP_ING/INPAT     36.257417   0.762258   36.182301   0.423543
SAT_02_ING/INPAT    2.001340   9.629865    2.369000   5.825700


In [6]:
ks_results = {}
for col in numeric_cols:
    stat, p_value = ks_2samp(df_covid[col].dropna(), df_synt[col].dropna())
    ks_results[col] = {'KS_statistic': stat, 'p_value': p_value}
print("\nKS-test Results:\n", pd.DataFrame(ks_results).T)


KS-test Results:
                    KS_statistic        p_value
EDAD/AGE               0.071349   4.574983e-04
UCI_DIAS/ICU_DAYS      0.928849  7.905050e-323
TEMP_ING/INPAT         0.200609   2.808563e-29
SAT_02_ING/INPAT       0.989276   0.000000e+00


In [7]:
# 3. Correlation matrices and difference
corr_real = df_covid[numeric_cols].corr()
corr_synth = df_synt[numeric_cols].corr()
corr_diff = corr_real - corr_synth
print("\nCorrelation Difference (real - synth):\n", corr_diff)


Correlation Difference (real - synth):
                    EDAD/AGE  UCI_DIAS/ICU_DAYS  TEMP_ING/INPAT  \
EDAD/AGE           0.000000           0.083681        0.089820   
UCI_DIAS/ICU_DAYS  0.083681           0.000000        0.321247   
TEMP_ING/INPAT     0.089820           0.321247        0.000000   
SAT_02_ING/INPAT  -0.013911          -0.119637        0.013560   

                   SAT_02_ING/INPAT  
EDAD/AGE                  -0.013911  
UCI_DIAS/ICU_DAYS         -0.119637  
TEMP_ING/INPAT             0.013560  
SAT_02_ING/INPAT           0.000000  


In [11]:
diagnosis_col = 'DIAG ING/INPAT'
drug_col = 'FARMACO/DRUG_NOMBRE_COMERCIAL/COMERCIAL_NAME'

top_diag_real = df_covid[diagnosis_col].value_counts().head(5).rename_axis('Diagnóstico').reset_index(name='Frecuencia Real')
top_diag_synth = df_synt[diagnosis_col].value_counts().head(5).rename_axis('Diagnóstico').reset_index(name='Frecuencia Sintética')

top_drug_real = df_covid[drug_col].value_counts().head(5).rename_axis('Fármaco').reset_index(name='Frecuencia Real')
top_drug_synth = df_synt[drug_col].value_counts().head(5).rename_axis('Fármaco').reset_index(name='Frecuencia Sintética')

print(top_diag_real)
print(top_diag_synth)
print(top_drug_real)
print(top_drug_synth)

           Diagnóstico  Frecuencia Real
0   COVID19 - POSITIVO             4062
1  COVID19 - PENDIENTE              414
          Diagnóstico  Frecuencia Sintética
0  COVID19 - POSITIVO                  1000
                            Fármaco  Frecuencia Real
0              DOLQUINE comp 200 mg             2028
1  MORFINA amp 1% 1 mL (10 mg/1 mL)              308
2      VENTOLIN aerosol 100 mcg/inh              290
3              BISOPROLOL comp 5 mg              268
4      HIDROXICLOROQUINA comp 200mg              202
                            Fármaco  Frecuencia Sintética
0              DOLQUINE comp 200 mg                   835
1  MORFINA amp 1% 1 mL (10 mg/1 mL)                   110
2                   ALCOHOL 70Âº 1L                    20
3             NORAGES amp 8 mg/4 mL                    19
4              BISOPROLOL comp 5 mg                    10


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Asegura estilos coherentes
sns.set(style="whitegrid")

# Figura 5.1: Histograma de edad (real vs sintético)
plt.figure(figsize=(10, 6))
bins = range(0, 101, 5)  # Rango de edades agrupado por intervalos de 5 años
plt.hist(df_covid['EDAD/AGE'], bins=bins, alpha=0.6, label='Real', color='blue', edgecolor='black')
plt.hist(df_synt['EDAD/AGE'], bins=bins, alpha=0.6, label='Sintético', color='orange', edgecolor='black')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()

